Chunking of Documents

In [1]:
#imports
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
import os

g:\Python Projects\2 AI concepts and patterns\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#preparing a text splitters

char_splitter = RecursiveCharacterTextSplitter(chunk_size=128,chunk_overlap=0)

In [3]:
directory = "./Documents"
fileNames = os.listdir(directory)
chunks = []
for file in fileNames:
    with open(f"{directory}/{file}") as f:
        filetext = f.read()
        texts = char_splitter.split_text(filetext)
        for chunkindex,text in enumerate(texts):
            chunks.append(Document(page_content=text,metadata={
                "chunkindex":chunkindex,
                "sourcefile":file
            }))

In [4]:
for chunk in chunks[:2]:
    print("------------")
    print("Page Content")
    print(chunk.page_content)
    print("Metadata")
    print(chunk.metadata)

------------
Page Content
# Development Team Internal Directory

## Team Scope

Handles:
Metadata
{'chunkindex': 0, 'sourcefile': 'Development Team.txt'}
------------
Page Content
* Feature development (frontend & backend)
* Bug fixing and issue resolution
* API development and integration
Metadata
{'chunkindex': 1, 'sourcefile': 'Development Team.txt'}


In [5]:
# if embeddings:
#     del embeddings

embeddings = HuggingFaceEmbeddings()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2114.42it/s]


In [17]:
emb = embeddings.embed_documents(chunks[0].page_content)
print(emb)

[[-0.057055991142988205, 0.05177615210413933, -0.03175172209739685, -0.02822735160589218, 0.07142482697963715, -0.019344601780176163, -0.04746702313423157, 0.04372545704245567, -0.029767438769340515, 0.029242120683193207, 0.08726590871810913, 0.049222543835639954, 0.018095267936587334, 0.05768929794430733, -0.04844367131590843, -0.010703126899898052, 0.03153146803379059, 0.04614533856511116, -0.005242527462542057, 0.009411669336259365, -0.0034075838048011065, 0.019546927884221077, 0.012306896038353443, 0.011264552362263203, -0.05656454712152481, 0.031810637563467026, 0.022501759231090546, -0.025371164083480835, 0.02880561165511608, -0.012655950151383877, -0.0683269277215004, -0.04287778213620186, 0.013329317793250084, -0.036988068372011185, 2.3603690806339728e-06, 0.029408007860183716, 0.031592417508363724, 0.011232960969209671, -0.05789298936724663, 0.02098570019006729, -0.003787481924518943, -0.05316530540585518, -0.012159292586147785, 0.03793807700276375, 0.004694936331361532, 0.033

In [6]:
vector_store = FAISS.from_documents(documents=chunks,embedding=embeddings)

Relevance
Relevance refers to how well a search result matches a user's query in terms of topic, intent, or content.  A relevant result directly addresses the user’s information need—e.g., for the query "jaguar", showing results about the animal, car, or NFL team depending on context. High relevance ensures users find accurate, useful information quickly. 

Diversity
Diversity ensures that search results cover multiple distinct aspects or perspectives of a query, especially when the intent is ambiguous or multifaceted.  For example, a diverse result set for "jaguar" would include content about the animal, the car brand, and the football team, avoiding redundancy and broadening coverage. 

Trade-off
Systems often balance relevance (returning the best-matching items) and diversity (showing varied, non-redundant items).  Techniques like Maximal Marginal Relevance (MMR) use a parameter λ to control this:

High λ → more diversity
Low λ → more relevance 
This prevents "filter bubbles" and improves user experience by exposing different viewpoints.

The expression λ(max_similarity) - (1-λ)(max_similarity) simplifies to:

(2λ - 1) * max_similarity

This represents a weighted transformation of the maximum similarity score, where:

When λ = 1, output = max_similarity
When λ = 0.5, output = 0
When λ = 0, output = -max_similarity

In [25]:
# retriever = vector_store.as_retriever(top=10)
mmr_retrieved_data=vector_store.max_marginal_relevance_search(
    k=3,
    lambda_mult=0.8,
    query="How much years of experience Ankit Sharma have?")
for i in mmr_retrieved_data:
    print("-----------")
    print(i.page_content)

-----------
---

### 2. Ankit Sharma
-----------
Contact: ankit.mehta@company.com
Priority Level: High
10. Pooja Deshmukh
Role: Junior IT Support
Experience: 2 years
-----------
Team Members (Rich Data for RAG)
1. Rahul Sharma
Role: Senior IT Support Engineer
Experience: 8 years


In [8]:
#ReRanker
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

# Your base retriever (e.g., FAISS)
base_retriever = vector_store.as_retriever(search_kwargs={"k": 5})

# Initialize BGE reranker
model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3")
compressor = CrossEncoderReranker(model=model, top_n=3)

# Wrap retriever
reranker = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

# Retrieve
docs = reranker.invoke("How much years of experience Ankit Sharma have?")   

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 1790.86it/s]


In [9]:
docs

[Document(id='ae11d181-b2be-4ac7-8b87-447b8b69329f', metadata={'chunkindex': 9, 'sourcefile': 'Development Team.txt'}, page_content='---\n\n### 2. Ankit Sharma'),
 Document(id='92c9deee-523f-4520-b27b-2d0416c14d68', metadata={'chunkindex': 35, 'sourcefile': 'IT Support Team.txt'}, page_content='Contact: ankit.mehta@company.com\nPriority Level: High\n10. Pooja Deshmukh\nRole: Junior IT Support\nExperience: 2 years'),
 Document(id='24a06b15-6dd1-454c-baf6-06d27385d27b', metadata={'chunkindex': 1, 'sourcefile': 'IT Support Team.txt'}, page_content='Team Members (Rich Data for RAG)\n1. Rahul Sharma\nRole: Senior IT Support Engineer\nExperience: 8 years')]

In [10]:
def get_surrounding_chunks(retrieved_chunk, all_chunks, window=5):
    idx = retrieved_chunk.metadata["chunkindex"]
    start = max(0, idx - window)
    end = min(len(all_chunks), idx + window + 1)
    return all_chunks[start:end]

In [11]:
# retrieved_chunks=retriever.invoke("How much years of experience Ankit Sharma have?")

In [12]:
context_chunks = []
for retrievedchunk in docs:
    context_chunks = context_chunks + get_surrounding_chunks(retrievedchunk,chunks)

In [13]:
print(len(context_chunks))
contextText = ""
contextText.join([f"Source:{content.metadata["sourcefile"]}, content : {content.page_content}" for content in context_chunks])

29


'Source:Development Team.txt, content : * Role: Senior Frontend Developer\n* Experience: 7 years\n* Expertise: React, UI performance, state managementSource:Development Team.txt, content : * Skill Level: Expert\n* Primary Keywords: UI bug, frontend issue, component not rendering, state issueSource:Development Team.txt, content : * Secondary Skills: accessibility, responsive design\n* Tools Used: React, Redux, Chrome DevToolsSource:Development Team.txt, content : * Desk Location: Floor 4, Bay F12\n* Availability: 10 AM â€“ 6 PMSource:Development Team.txt, content : * Contact: [rohan.mehta@company.com](mailto:rohan.mehta@company.com)\n* Priority Level: HighSource:Development Team.txt, content : ---\n\n### 2. Ankit SharmaSource:Development Team.txt, content : * Role: Backend Developer\n* Experience: 5 years\n* Expertise: Node.js, APIs, database handling\n* Skill Level: ExpertSource:Development Team.txt, content : * Primary Keywords: API error, backend issue, server error, database issue\n

In [ ]:
if embeddings:
    del embeddings
if model:
    del model
# import torch
# print(torch.cuda.is_available()) 